In [7]:
import numpy as np
import h5py
import time

import astropy.units as u
import astropy.cosmology.units as cu
import astropy.constants as const

import raytrace
import sys

rng = np.random.default_rng(seed=1)

with h5py.File("outputs/dm_proto.hdf5", "r") as f:
    halo = f['halo_3238091']
    com = halo['GroupCM'][:]
    # if not repeat and 'Group_Fgas_Crit200' in halo.keys():
    #     print('halo already calculated')
    #     sys.exit(0)

    r200 = halo['Group_R_Crit200'][()]
    r500 = halo['Group_R_Crit500'][()]
    
    m200 = halo['Group_M_Crit200'][()]
    m500 = halo['Group_M_Crit500'][()]
    subs = halo['subhalos']
    sub0_cm = np.array([ subs['cm_x'][0], subs['cm_y'][0], subs['cm_z'][0] ]).T

with h5py.File(f'tng_cache/snap_099/cutout_3238091.hdf5', 'r') as f:

    gas_pos = raytrace.unwrap(f['PartType0/Coordinates'][:] - sub0_cm) # ckpc
    gas_mass = f['PartType0/Masses'][:] # Msun
    windy = f['PartType4/GFM_StellarFormationTime'][:] < 0
    wind_mass = f['PartType4/Masses'][windy] # Msun
    wind_pos = raytrace.unwrap(f['PartType4/Coordinates'][windy] - sub0_cm) # Msun
    
    eta_e = f['PartType0/ElectronAbundance'][:] # fraction
    X_H = f['PartType0/GFM_Metals'][:,0] # fraction
    gas_m_e = gas_mass * eta_e * X_H # Msun

r = np.linalg.norm(gas_pos, axis=-1)
rwind = np.linalg.norm(wind_pos, axis=-1)
fgas200 = np.sum(gas_mass[r < r200])/m200
fgaswind500 = np.sum(wind_mass[rwind < r500])/m500
fgas500 = np.sum(gas_mass[r < r500])/m500
fe200 = np.sum(gas_m_e[r < r200])/m200
fe500 = np.sum(gas_m_e[r < r500])/m500

print(fgas500, fgaswind500)

0.14568504088829415 1.0990359567260859e-07
